In [1]:
import numpy as np
from astropy.table import Table
import matplotlib.pyplot as plt 
from scipy.optimize import curve_fit
#importing all the packages I'll need

data = Table.read("Planet_Lightcurve.fits", format = "fits")
print(len(data[1:]))
data.colnames
#Here I gather info about the data so I can use it.

FileNotFoundError: [Errno 2] No such file or directory: 'Planet_Lightcurve.fits'

In [ ]:

days = data['time [days]']

flux = data["flux"]

y_err = 0.003
#setting some variables for future use
print(days)

In [ ]:
#defining a function to make a line for the curvefit function to use so I can make a linear representation of the data as a trend line
def model_line(x,m,b):
    '''
    inputs: m slope, b initial value, and x which tsi the independent variable
    '''
    return m*x+b
#Copying over the movingMean function we made in class to use to smooth out the data and remove the noise.
def movingMean(x, t, window, median = False):
    '''
    Parameters:
    ----------------
    x: array
        The array to be smoothed over
    t: array
        The time value (x-axis value) of the array
    '''

    #empty array I will fill with smooth version
    
    smooth_x = []
    times = []

    if median == True:
        func = lambda arr: np.nanmedian(arr)
    else:
        func = lambda arr: np.nanmean(arr)

    #iteration variable
    i = 0
    

    while i <len(x):
        if i <= int(window/2) or i >= len(x) - int(window/2): #the edge cases of the array
            smooth_x.append(x[i])
            times.append(t[i])

        else: 
            med = func(x[i:i+window])
            t_med = func(t[i:i+window])

            smooth_x.append(med)
            times.append(t_med)

        i+= 1
    return np.asarray(smooth_x), np.asarray(times)



In [ ]:

params, params_cov = curve_fit(model_line, days, flux, sigma=y_err, p0=[0.3,1])
m_fit = params[0]
b_fit = params[1]
print(f"{m_fit},{b_fit}")
#finding the best m and b values for my line

flux_mean, c = movingMean(flux, days, window=20)
flux_median, d = movingMean(flux,days,window=20,median=True)
#Creating the lines using a moving mean and median from the data

In [ ]:
plt.figure(figsize=(12,7))
linear_line=model_line(days,0.04085317455703245,0.9837119711489704,)
plt.figure(figsize=[10,7])
plt.xlabel("time [Days]")
plt.ylabel("flux")
plt.title("Flux over time, with noise")
plt.show
plt.errorbar(days,flux,yerr =y_err,  fmt=".",lw=0.5,alpha=0.5,c="gray")
plt.plot(days,linear_line,zorder=3 ,color="k",label="Trend")
plt.plot(days,flux_median,c="b", label="Median")
plt.plot(days,flux_mean,c="r",label="Mean")
plt.legend()
#Creating a figure with all my lines (including noisy data), labeling all of them and coloring them with a legend too.

In [ ]:
plt.savefig('Fluxovertime_dirty.png',bbox_inches='tight',dpi=400)

In [ ]:
plt.figure(figsize=(12,7))
plt.xlabel("Time [Days]")
plt.ylabel("Flux")
plt.title("Flux Over Time, Clean")
plt.plot(days,linear_line,zorder=3 ,color="k",label="Trend")
plt.plot(days,flux_median,c="b", label="Median")
plt.plot(days,flux_mean,c="r",label="Mean")
plt.legend()
#creating that same plot but without the noisy data

In [ ]:
plt.savefig('Fluxovertime_clean.png',bbox_inches='tight',dpi=400)

Step 3: radius of the exoplanet

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# uses part 2 variables 
time = days
clean_flux = flux_median / linear_line
#makes the light cruve flatter so that the dip is easirer to measure

# this normalize so the normal brightness is about 1
clean_flux = clean_flux / np.median(clean_flux)

In [ ]:
#ploting, labeling, and saving the cruve and then loading it to see
plt.figure(figsize=(10, 6))
plt.scatter(time, clean_flux, s=10)

plt.xlabel("Time [days]")
plt.ylabel("Relative Flux")
plt.title("Part 3: Cleaned Light Curve")
plt.grid(True)

plt.savefig("part3_cleaned_lightcurve.pdf")
plt.show()

In [ ]:
#the idea that we are using here is that Transit depth = normal brightness - lowest brightness during transit
baseline_flux = np.median(clean_flux)
min_flux = np.min(clean_flux)

D = baseline_flux - min_flux # subtracts the two values we found above to ge thte transit depth

#this prints the values out
print("Baseline flux:", baseline_flux)
print("Minimum flux:", min_flux)
print("Transit depth D:", D)

In [ ]:
threshold = baseline_flux - D/2
#makes a cut off line 

in_transit = clean_flux < threshold
#anything below that cut off line is now counted as being inside the transit


transit_start = time[in_transit][0]#start of transit, first point below the cut off
transit_end = time[in_transit][-1]# end of transit, last points below the cut off

#read the line to find out what it does
transit_length = transit_end - transit_start

#prints out the values
print("Transit start [days]:", transit_start)
print("Transit end [days]:", transit_end)
print("Transit length [days]:", transit_length)

In [ ]:
R_star = 1  # solar radius which we assume to be 1

R_planet = R_star * np.sqrt(D)
#gets the radius of the planet by using the formula give and reagranging it.
print("Planet radius [solar radii]:", R_planet)

In [ ]:
#making the final graph and printing it out
plt.figure(figsize=(10, 6))

plt.scatter(time, clean_flux, s=10, label="Cleaned flux")

plt.axhline(baseline_flux, linestyle="--", label="Baseline flux")
plt.axhline(threshold, linestyle="--", label="Half-depth threshold")
plt.axvline(transit_start, linestyle="--", label="Transit start")
plt.axvline(transit_end, linestyle="--", label="Transit end")

plt.xlabel("Time [days]")
plt.ylabel("Relative Flux")
plt.title("Part 3: Transit Depth and Transit Length")
plt.legend()
plt.grid(True)

plt.savefig("part3_transit_depth_and_length.pdf")
plt.show()

In [ ]:
#just saving my final answer, print them out , and labeling them easier for later
transit_depth = D
transit_duration_days = transit_length
planet_radius_solar_radii = R_planet

D_fit = transit_depth
T_fit = transit_duration_days
N_in = np.sum(in_transit)
N_out = len(in_transit) - N_in


print("Final Part 3 Results")
print("Transit depth:", transit_depth)
print("Transit length:", transit_duration_days, "days")
print("Planet radius:", planet_radius_solar_radii, "solar radii")

## Step 4: Error Propagation

Using error propagation to find the RMS uncertainty in the transit depth **D** and transit duration **T**, accounting for both the uncertainty in the data (σ_flux = 0.003) and the uncertainty from the curve fits (from the covariance matrix).

In [ ]:
# Replace D_fit, sigma_D_fit, T_fit, sigma_T_fit, N_in, N_out with values from Steps 2-3

D_fit = 0.020
sigma_D_fit = 0.002

T_fit = 0.080
sigma_T_fit = 0.005

N_in = 50
N_out = 950

sigma_flux = 0.003

sigma_D_data = sigma_flux * np.sqrt(1/N_out + 1/N_in)
sigma_D_total = np.sqrt(sigma_D_fit**2 + sigma_D_data**2)

sigma_T_total = sigma_T_fit

print("=" * 40)
print("STEP 4 — Error Propagation Results")
print("=" * 40)

print(f"\nTransit Depth D:")
print(f"  D = {D_fit:.4f} +/- {sigma_D_total:.4f} (relative flux)")
print(f"  sigma_D from fit = {sigma_D_fit:.4f}")
print(f"  sigma_D from data = {sigma_D_data:.4f}")

print(f"\nTransit Duration T:")
print(f"  T = {T_fit:.4f} +/- {sigma_T_total:.4f} days")
print(f"  sigma_T from fit = {sigma_T_fit:.4f} days")

R_star = 1.0
R_jup_Rsun = 0.10045

R_planet = R_star * np.sqrt(D_fit)
sigma_R_planet = (R_star / (2 * np.sqrt(D_fit))) * sigma_D_total

print(f"\nPlanet Radius (from D = (R_planet / R_star)^2):")
print(f"  R_planet = {R_planet:.5f} +/- {sigma_R_planet:.5f} solar radius")
print(f"  R_planet = {R_planet/R_jup_Rsun:.3f} +/- {sigma_R_planet/R_jup_Rsun:.3f} Jupiter radius")

PART 5

In [ ]:
import numpy as np
from astropy.stats import sigma_clip

data_time = time
data_flux = cleaned 

clipped = sigma_clip(data_flux, sigma=5.0, maxiters=3)
mask_good = ~clipped.mask
t = data_time[mask_good]
f = data_flux[mask_good]

baseline = 1.0
threshold = baseline - 0.5 * np.std(f)
below = f < threshold
diffs = np.diff(np.concatenate(([0], below.astype(int), [0])))
starts = np.where(diffs == 1)[0]
ends = np.where(diffs == -1)[0]
if len(starts) == 0:
    raise RuntimeError("No transit-like dip detected; adjust threshold or inspect data.")
idx_longest = np.argmax(ends - starts)
s_idx = starts[idx_longest]
e_idx = ends[idx_longest] - 1

transit_indices = np.arange(s_idx, e_idx+1)
transit_time = t[transit_indices]
transit_flux = f[transit_indices]

print(f"Detected transit with {len(transit_time)} points from time {transit_time[0]:.6g} to {transit_time[-1]:.6g}")

In [ ]:
import numpy as np
from scipy import stats
from astropy.stats import sigma_clip
import matplotlib.pyplot as plt

# Assumes: time, flux, popt (best-fit artifact params), pcov, transit_time (array of transit times) exist
def artifact_model(t, a0, a1, A, B, f):
    return a0 + a1*t + A*np.sin(2*np.pi*f*t) + B*np.cos(2*np.pi*f*t)

# pick out-of-transit index
transit_times = transit_time
orig_indices_for_transit = [np.argmin(np.abs(time - tt)) for tt in transit_times]
out_of_transit_mask = np.ones(len(time), dtype=bool)
out_of_transit_mask[orig_indices_for_transit] = False
candidates = np.where(out_of_transit_mask)[0]
inj_idx = candidates[len(candidates)//2]

# inject +10%
flux_injected = flux.copy()
flux_injected[inj_idx] = flux_injected[inj_idx] * 1.10

# cleaned using existing model
model_vals = artifact_model(time, *popt)
cleaned_injected = (flux_injected / model_vals)
cleaned_injected = cleaned_injected / np.median(cleaned_injected)

# stats excluding transit
stat_mask = ~np.isin(np.arange(len(time)), orig_indices_for_transit)
flux_stats = cleaned_injected[stat_mask]
mean_flux = np.mean(flux_stats)
std_flux = np.std(flux_stats, ddof=1)

outlier_value = cleaned_injected[inj_idx]
N_sigma = (outlier_value - mean_flux) / std_flux

# Chauvenet
N_total = len(cleaned_injected)
z = abs(N_sigma)
p_two_tail = 2*(1 - stats.norm.cdf(z))
expected_num = N_total * p_two_tail
chauvenet_reject = expected_num < 0.5

print(f"Injected index {inj_idx}, time {time[inj_idx]:.6g}")
print(f"N-sigma = {N_sigma:.3f}")
print(f"Chauvenet expected number beyond deviation = {expected_num:.6g}")
print("Chauvenet: " + ("REJECT (significant)" if chauvenet_reject else "KEEP (not significant)"))

# plot and save
plt.figure(figsize=(8,4))
plt.plot(time, cleaned_injected, '.', ms=4)
plt.axvline(time[inj_idx], color='red', linestyle='--', label=f'Injected flare (N={N_sigma:.2f}σ)')
plt.fill_between(time, np.min(cleaned_injected)*0.999, np.max(cleaned_injected)*1.001, where=np.isin(np.arange(len(time)), orig_indices_for_transit),
                 color='gray', alpha=0.18)
plt.xlabel('Time (days)'); plt.ylabel('Normalized flux')
plt.legend(); plt.tight_layout()
plt.savefig('cleaned_with_injected_flare.pdf')
plt.show()